# INESC TEC Uncertainty Framework - Colab Pro+ Report Runner
Este notebook é o ponto de entrada oficial para a execução assíncrona do projeto.

**ATENÇÃO:** O repositório irá clonar a *tag* `report-run-freeze` em vez da branch `main`.

In [ ]:
# 1. Mount Drive, Setup Environment and Clone Frozen Tag
from google.colab import drive
drive.mount('/content/drive')
from google.colab import userdata
github_token = userdata.get('GITHUB_TOKEN')
!git clone --branch report-run-freeze https://{github_token}@github.com/AlexandreCGodinho/inesctec-uncertainty-framework.git
%cd inesctec-uncertainty-framework
!pip install -r requirements.txt
!pip install vllm

In [ ]:
# 2. Start vLLM Server in Background
import subprocess
import time

vllm_process = subprocess.Popen([
    "python3", "-m", "vllm.entrypoints.openai.api_server",
    "--model", "google/gemma-4-12b-it",
    "--dtype", "bfloat16",
    "--gpu-memory-utilization", "0.90"
])

# Wait for server to be ready
time.sleep(60)

In [ ]:
# 3. Execute Report Run (25-Hour Production Run)
# Para o teste rápido de 20 min (micro-smoke), mude --output-dir para "/content/temp_test" e adicione --sample-size 2
!python3 -m src.runner --output-dir "/content/drive/MyDrive/report_run_a100"

In [ ]:
# 4. Audit & Verify Manifest
import json
from pathlib import Path

manifest_path = Path('/content/drive/MyDrive/report_run_a100/manifest.json')
if manifest_path.exists():
    with open(manifest_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    print('=== MANIFEST AUDIT ===')
    print('FinQA Commit:', data.get('finqa_commit', 'MISSING'))
    print('Runs:', list(data.get('runs', {}).keys()))
else:
    print('Manifest not found at:', manifest_path)